# N042 · 二分查找与左右边界

**目标：** 用统一半开区间维护搜索不变量。

**先修：** N013, N025, N037。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** lower_bound/upper_bound；不存在；重复值；中点与边界推进。

**配套讲解来源：** [同名逐章意见](../../comment/042_binary_search_bounds.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的问题是：**在一个从小到大排好序的数组里定位一个值 x**。比"在不在"更进一步，我们要能回答三种问法：x 第一个出现在哪（左边界）？x 最后出现在哪（右边界）？x 不在的话应该插在哪（插入位置）？有重复元素时这三问尤其容易写错，本章用一套统一的框架（半开区间 + 谓词）一次搞定。

先看具体到数字的例子（“运行示例”部分/“自动测试”部分 的真实数据）：

- 输入：`a = [1,2,2,4]`，`x = 2`
- 输出：`lower_bound(a,2) = 1`（第一个 ≥2 的位置）、`upper_bound(a,2) = 3`（第一个 >2 的位置）、`search_range(a,2) = [1,2]`（2 出现的区间是从下标 1 到下标 2）
- 为什么是这些值？数组里两个 2 分别坐在下标 1 和 2；"第一个不小于 2 的位置"正是下标 1；"第一个大于 2 的位置"是下标 3（那里放的是 4）；所以 2 的出现区间是 [1, 3−1] = [1, 2]。

再看 x 不在数组里的情形：`x = 3` 时，`lower_bound(a,3) = 3`——下标 0..2 的元素都小于 3、下标 3 的元素大于 3，所以"第一个 ≥3 的位置"是 3，这同时就是 3 的插入位置（插进去仍有序）；而 `search_range(a,3)` 返回 `[-1,-1]`，因为 3 根本没出现。

三种问法在同一组数据上的对照（a=[1,2,2,4]）：

| 问法 | 用的函数 | x=2 时 | x=3 时 |
|---|---|---|---|
| 第一个 ≥x 的位置（即插入位） | lower_bound | 1 | 3 |
| 第一个 >x 的位置 | upper_bound | 3 | 3 |
| x 的出现区间 | search_range | [1,2] | [-1,-1] |

注意 x=3 那一列：lower_bound 和 upper_bound 都返回 3，但两句话含义不同——前者说"3 应插在下标 3"，后者说"第一个大于 3 的元素在下标 3（那里是 4）"，两种视角在这一个点上恰好重合。

## 2. 基础知识：先读懂这些词

- **半开区间 [lo, hi)**：本章统一的区间约定——含 lo 不含 hi。它的好处是"区间长度 = hi − lo"永远成立、"空区间"就是 lo==hi，边界判断干净。len(a) 本身就是"数组末尾再往后一格"的天然 hi。
- **谓词（predicate）**：一个对每个位置回答"是/否"的条件。二分的本质是：谓词在有序数组上从某处开始恒为"是"、之前恒为"否"，二分找的就是这条分界线。lower_bound 的谓词是 a[i] ≥ x，upper_bound 的谓词是 a[i] > x。
- **lower_bound（左边界）**：第一个满足 a[i] ≥ x 的下标 i。若所有元素都小于 x，返回 len(a)。它同时就是"x 的插入位置"。
- **upper_bound（右边界之后）**：第一个满足 a[i] > x 的下标 i。若所有元素都不大于 x，返回 len(a)。它与 lower_bound 的差就是 x 出现的个数。
- **单调性**：数组有序保证了谓词的"前面全是否、后面全是是"结构——这是二分每步能扔掉一半的前提。数组无序时这套框架不成立。
- **不存在（absent）**：x 不在数组中时，lower_bound 指向插入位置而非有效命中；判断"存在"必须再补一枪：位置没越界且该位置的值确实等于 x。
- **重复值**：多个 x 连成一段，lower_bound/upper_bound 恰好夹住这一段的左右两侧，search_range 由两者拼出。
- **中点与边界推进**：mid = (lo+hi)//2 取偏左中点；谓词为"否"时 lo = mid+1（mid 自己已被否定，可放心扔掉），谓词为"是"时 hi = mid（mid 可能就是答案，不能扔，只能把右界收到它）。这两条更新规则保证区间每轮严格变短、且候选答案永远不被丢出区间。

用 a=[1,2,2,4]、x=2 把谓词结构看真切：对 lower_bound 的谓词 a[i]≥x，下标 0、1、2、3 上依次得到"否、是、是、是"——前面一段全否、后面一段全是，中间只有一条分界；二分要找的就是第一个"是"的位置。换成 upper_bound 的谓词 a[i]>x，答案变成"否、否、否、是"，分界右移了一格，这正是重复值被整段跳过的原因。

## 3. 思路推导：从小例子开始

- **Step 1：定区间约定。** 搜索区间取半开 [lo, hi)，初始 lo=0、hi=len(a)。所有"还没排除的答案位置"都在这个窗口里。
- **Step 2：写谓词。** 要左边界就用"a[i] ≥ x 吗"，要右边界就用"a[i] > x 吗"。两个谓词只差一个等号。
- **Step 3：取中点、问谓词、砍一半。** mid=(lo+hi)//2。若 a[mid] 不满足谓词（说明答案在 mid 右边），lo=mid+1；若满足（mid 可能是答案），hi=mid。每轮窗口长度至少砍半。
- **Step 4：循环到窗口空。** lo==hi 时窗口只剩一个位置，它就是分界线，返回 lo。
- **Step 5：组合出区间查询。** search_range 先取 left=lower_bound(a,x)；若 left 已到数组末尾或 a[left] 不等于 x，说明 x 不存在，返回 [-1,-1]；否则返回 [left, upper_bound(a,x)−1]。

**手算演示一：`lower_bound([1,2,2,4], 2)`（“运行示例”部分 表格的原始数据，一轮一行对照）。**

| lo | hi（不含） | mid | a[mid] | 判断与推进 |
|---|---|---|---|---|
| 0 | 4 | 2 | 2 | a[mid]<x？2<2 不成立 → hi=2 |
| 0 | 2 | 1 | 2 | 2<2 不成立 → hi=1 |
| 0 | 1 | 0 | 1 | 1<2 成立 → lo=1 |

lo=hi=1，窗口空，返回 1。第一张表正是 “运行示例”部分 用 show_table 展示的三行 (lo, hi, mid, a[mid])。体会窗口怎么收缩：第一轮 mid=2 本身就是个 2，但"它是第一个 2 吗"不知道，所以只能把右界收到 2（留着它），继续往左搜——这就是"hi=mid 不扔答案"的含义。

**手算演示二：`upper_bound([1,2,2,4], 2)`。** 谓词换成 a[mid] ≤ x：lo=0,hi=4：mid=2，a[2]=2 ≤ 2 成立 → lo=3；lo=3,hi=4：mid=3，a[3]=4 ≤ 2 不成立 → hi=3。lo=hi=3，返回 3。

**手算演示三：`search_range([1,2,2,4], 3)`。** lower_bound 得 3（手算：mid=2 时 2<3 → lo=3；mid=3 时 4<3 不成立 → hi=3），但 left=3 已等于 len(a)，判定不存在，返回 [-1,-1]。插入位置信息（3）与存在性判定（不存在）就这样被同一套工具分开表达。

三段手算合起来有一条实操经验：写二分之前，先在纸上把每一轮的 (lo, hi, mid, a[mid]) 列成 “运行示例”部分 那样的表，预测完再运行对照。大多数二分 bug（越界、死循环、差一）在三轮手算里就会现形，比盯着代码发呆快得多——这也是每章"小实例与可视化"一节让你先手算再运行的本意。

## 4. 核心代码：lower_bound

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def lower_bound(a, x):
    lo, hi = (0, len(a))
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo
```

### 逐段读懂代码

### 函数 1：`lower_bound` —— 第一个 ≥x 的位置

```python
def lower_bound(a, x):
    lo, hi = (0, len(a))
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo
```

- `lo,hi=0,len(a)`：初始窗口是整个数组。空数组时 lo=hi=0，循环不进，直接返回 0——"空数组的插入位置是 0"，自然正确。
- `while lo<hi:`：窗口非空就继续。半开约定下这就是"长度 > 0"。
- `mid=(lo+hi)//2`：取偏左中点。当 lo<hi 时必有 lo ≤ mid < hi，这一性质是下面两条更新能严格缩短窗口的根基。
- `if a[mid]<x: lo=mid+1`：a[mid] 比 x 小，mid 这个位置不可能是答案（谓词 a[i]≥x 在它这里不成立），连 mid 一起扔掉，左界推到 mid+1。写 mid+1 而不是 mid，正是为了避免 mid 被反复审、窗口不缩小的死循环（练习 2 专门研究反例）。
- `else: hi=mid`：a[mid] ≥ x，mid 满足谓词，但它是不是"第一个"还不知道，只能把右界收到 mid——mid 留在窗口里当候选。
- `return lo`：窗口缩空时 lo==hi，正是"否/是"的分界位置。

### 函数 2：`upper_bound` —— 第一个 >x 的位置

```python
def upper_bound(a, x):
    lo, hi = (0, len(a))
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo
```

与 lower_bound 只差一个字符：判断条件从 `a[mid]<x` 变成 `a[mid]<=x`。含义是"等于 x 的位置也算'还不够'，答案要更靠右"。就这一个等号，让重复的 x 整段被跳过去，得到第一个严格大于 x 的位置。

### 函数 3：`search_range` —— x 的出现区间

```python
def search_range(a, x):
    left = lower_bound(a, x)
    if left == len(a) or a[left] != x:
        return [-1, -1]
    return [left, upper_bound(a, x) - 1]
```

- `left=lower_bound(a,x)`：先找第一个 ≥x 的位置，它是"若 x 存在，最左出现处"的唯一候选。
- `if left==len(a) or a[left]!=x: return [-1,-1]`：存在性判定要防两种失效——left 走到了数组末尾（所有元素都小于 x），或者 left 上的元素根本不是 x（x 介于两元素之间，left 指向的是"第一个大于 x 的位置"）。`or` 短路求值保证先查越界再取 a[left]，顺序不能颠倒。
- `return [left,upper_bound(a,x)-1]`：x 存在时，最左出现是 left；upper_bound 是"第一个 >x 的位置"，再退一格就是最右出现。

## 5. 分段实现：按顺序运行

**本章接口：** `lower_bound(a, x)`、`upper_bound(a, x)`、`search_range(a, x)`

### lower_bound

In [1]:
def lower_bound(a, x):
    lo, hi = (0, len(a))
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

### upper_bound

In [2]:
def upper_bound(a, x):
    lo, hi = (0, len(a))
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo

### search_range

In [3]:
def search_range(a, x):
    left = lower_bound(a, x)
    if left == len(a) or a[left] != x:
        return [-1, -1]
    return [left, upper_bound(a, x) - 1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,2,4]; lo=0; hi=len(a); rows=[]
while lo<hi:
    mid=(lo+hi)//2; rows.append((lo,hi,mid,a[mid]))
    if a[mid]<2: lo=mid+1
    else: hi=mid
show_table(['lo','hi（不含）','mid','a[mid]'],rows)

lo,hi（不含）,mid,a[mid]
0,4,2,2
0,2,1,2
0,1,0,1


## 7. 正确性、适用条件与复杂度

lower_bound维持lo前所有元素<x、hi起所有元素≥x。更新规则保留该性质，长度严格下降，lo=hi时就是分界位置。

**代价：** O(log(n+1))时间、O(1)辅助空间。测试验证结果；复杂度结论来自区间收缩，不来自固定计时门槛。

### 展开理解

**为什么 lower_bound 返回的正是分界位置？** 用一句始终成立的话来撑住整个循环：**在循环的任何时刻，下标 lo 之前的所有元素都严格小于 x，下标 hi 及之后的元素都大于或等于 x；而答案——第一个 ≥x 的位置——始终落在 [lo, hi] 里。** 开头这句话成立吗？成立：初始 lo=0，"0 之前"是空的，空话为真；初始 hi=len(a)，"len(a) 及之后"也是空的，同样为真。每轮循环会不会把它弄坏？看两条更新：当 a[mid]<x 时，mid 位置的值小于 x，于是 mid 和它左边的全部位置都不可能是分界（它们都 <x），把 lo 推到 mid+1 之后，"lo 之前全 <x"依然成立；当 a[mid]≥x 时，mid 位置的值满足条件，mid 及其右边全是"≥x"的领地，把 hi 收到 mid 之后，"hi 起全 ≥x"依然成立——而且 mid 被留在窗口里，答案没有被扔掉。循环会停吗？会：lo<hi 保证 lo ≤ mid < hi，于是 lo=mid+1 至少把 lo 推进一格、hi=mid 至少把 hi 拉回一格，窗口长度每轮至少减一，有限轮后必空。窗口空时 lo==hi，此时"lo 之前全 <x"与"hi 起全 ≥x"在同一处会合——这个位置既是"小于 x 部分的右边界外一格"，又是"≥x 部分的起点"，正是要找的分界。upper_bound 的论证一模一样，只是"≥x"换成">x"、"<x"换成"≤x"。

**复杂度是多少？** 每轮窗口至少砍半，n 个元素最多问 ⌈log₂(n+1)⌉ 次；辅助空间只有 lo、hi、mid 三个变量，O(1)。拿数字感受：n = 10⁶ 的一百万个数组只要约 20 次比较，n = 10⁹（理论上的超大数组）也只要 30 次——这就是 “正确性与复杂度”部分 说"复杂度结论来自区间收缩，不来自固定计时门槛"的意思：快是因为每轮必扔一半这个结构性事实，不是因为测得快。另外别忘了前提：**数组必须有序**，无序数组上谓词的分界结构不存在，这套循环毫无意义。

再补一个初学者常问的细节：为什么 mid=(lo+hi)//2 的写法不会死循环？只要 lo<hi，整数除法保证 lo ≤ mid < hi。于是 hi=mid 这一步必然让 hi 至少回退一格（mid 严格小于旧 hi）；lo=mid+1 这一步必然让 lo 至少前进一格（mid+1 严格大于旧 lo）。两个变量都只能朝对方走、谁也不会原地踏步，窗口长度每轮至少减一，循环必然终止。练习 2 会让你亲手写一个破坏这一性质的版本，看看"mid 不动"是怎么把循环卡死的。

顺带一提：这里的 mid 永远落在 [0, len(a)) 之内，所以取 a[mid] 不会越界；有些写法把 hi 初始化成 len(a)-1（闭区间约定），中点就可能顶到数组末尾、边界分支也得多写一条——这就是本章坚持半开约定的现实收益。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看：

- `assert lower_bound([1,2,2,4],2)==1 and upper_bound([1,2,2,4],2)==3`：**正常值测试（重复元素的手算锚点）**。两个 2 挤在中间，左边界 1、右边界之后 3，第三节已逐轮手推。这条专抓"等号位置写错"——谓词差一个 =，两个答案就会互换。
- `assert lower_bound([],2)==0`：**边界值测试（空数组）**。空数组任何值的插入位置都是 0，同时验证初始 lo=hi 时循环体根本不执行、不越界。
- 穷举块（`for n in range(8): for a in combinations_with_replacement([0,1,2],n): for x in range(-1,4)`）：**交叉验证测试（标准库当裁判 + 穷举小规模）**。这里有个巧思：`combinations_with_replacement([0,1,2], n)` 生成的正是所有长度 n、取值 0..2 的**非降序列**——恰好是二分的合法输入前提，穷举不漏一种形态；x 从 −1 取到 3，覆盖"比所有元素小""正好是某元素""介于两元素之间""比所有元素大"四种关系。三件事逐一对照：`lower_bound(a,x)==bisect_left(a,x)`、`upper_bound(a,x)==bisect_right(a,x)`（Python 标准库 bisect 就是这两个函数的官方实现，拿来当裁判）；`search_range(a,x)` 与"实际下标列表的首尾"（`[indices[0],indices[-1]]`，不存在则 [-1,-1]）一致。穷举总量是长度 0..7 的全部非降数组 × 5 个 x，规模小但把"重复、缺失、两端"全部踩遍。

| 断言 | 类型 | 它防的是哪类 bug |
|---|---|---|
| lower_bound/upper_bound([1,2,2,4],2) | 正常值（重复元素锚点） | 谓词里等号位置写错 |
| lower_bound([],2)==0 | 边界（空数组） | 初始窗口或返回值错 |
| 穷举非降数组 × 5 种 x 对拍 bisect | 穷举交叉验证 | 边界推进写错、search_range 拼接错 |

In [5]:
from bisect import bisect_left, bisect_right

from itertools import combinations_with_replacement

assert lower_bound([1, 2, 2, 4], 2) == 1 and upper_bound([1, 2, 2, 4], 2) == 3

assert lower_bound([], 2) == 0

for n in range(8):
    for a in combinations_with_replacement([0, 1, 2], n):
        for x in range(-1, 4):
            assert lower_bound(a, x) == bisect_left(a, x)
            assert upper_bound(a, x) == bisect_right(a, x)
            indices = [i for i, v in enumerate(a) if v == x]
            assert search_range(a, x) == ([indices[0], indices[-1]] if indices else [-1, -1])

print('N042: 所有本章断言通过')

N042: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 从lower_bound派生精确查找。

**练习 2：** 写出mid不变导致死循环的反例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（从 lower_bound 派生精确查找）**：方向提示——"在不在"只要在 lower_bound 后面补一步验证：`pos=lower_bound(a,x)`，当 `pos<len(a)` 且 `a[pos]==x` 时 x 存在且下标就是 pos，否则不存在。想清楚为什么这一枪必须打：lower_bound 只承诺"第一个 ≥x"，x 缺席时它指向的是下一个更大的元素（或末尾），不验证就会把"插入位置"误报成"命中"。手算示例用 [1,2,2,4]、x=3：lower_bound 返回 3，a[3]=4≠3，判不存在。再用本章接口或 `x in a` 对拍验证。
- **练习 2（写出 mid 不变导致死循环的反例）**：方向提示——把更新规则故意写坏：谓词成立时不写 `lo=mid+1` 而写 `lo=mid`。反例取 `a=[1,3], x=1`，套 lower_bound 的判断：lo=0, hi=2：mid=1，a[1]=3 < 1 不成立 → hi=1；此后 lo=0, hi=1：mid=0，a[0]=1 < 1 不成立 → hi=0，循环倒是结束了——所以更干净的构造是把坏分支放在会触发的一侧，例如对 upper_bound 的条件写坏成 `a[mid]<=x: lo=mid`：lo=0, hi=2：mid=1，a[1]=3 ≤ 1 不成立 → hi=1；lo=0, hi=1：mid=0，a[0]=1 ≤ 1 成立 → lo=mid=0——窗口长度停在 1，永远循环。问题根源一句话：当 hi−lo==1 时 (lo+hi)//2 == lo，若此时执行 lo=mid，lo 原地踏步、窗口不再缩短。把这段手推写进报告，并总结防御办法：要么 lo=mid+1（越过 mid），要么改用偏右中点 (lo+hi+1)//2 配 lo=mid，保证每轮必有变量移动。

两道练完的自查标准：你不看资料能默写 lower_bound 与 upper_bound，并说清两者只差一个等号、差在哪一侧的语义上；同时能解释 search_range 里那行存在性判定为什么两个条件缺一不可（越界检查在前、相等检查在后，顺序也不能换）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def lower_bound(a, x):
    lo, hi = (0, len(a))
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

def upper_bound(a, x):
    lo, hi = (0, len(a))
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo

def search_range(a, x):
    left = lower_bound(a, x)
    if left == len(a) or a[left] != x:
        return [-1, -1]
    return [left, upper_bound(a, x) - 1]

# 示例与回归测试
from bisect import bisect_left, bisect_right

from itertools import combinations_with_replacement

assert lower_bound([1, 2, 2, 4], 2) == 1 and upper_bound([1, 2, 2, 4], 2) == 3

assert lower_bound([], 2) == 0

for n in range(8):
    for a in combinations_with_replacement([0, 1, 2], n):
        for x in range(-1, 4):
            assert lower_bound(a, x) == bisect_left(a, x)
            assert upper_bound(a, x) == bisect_right(a, x)
            indices = [i for i, v in enumerate(a) if v == x]
            assert search_range(a, x) == ([indices[0], indices[-1]] if indices else [-1, -1])

print('N042: 所有本章断言通过')

N042: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **704. Binary Search（二分查找）**：有序数组里找 target 的下标，不存在返回 −1。这题练的是练习 1 的内容——精确查找就是"lower_bound + 越界与相等验证"，本章框架直接套用。
- **34. Find First and Last Position of Element in Sorted Array（排序数组中查找元素的第一个和最后一个位置）**：这就是 `search_range` 的原题（要求 O(log n)，重复段很长也不许线性扫）。练的是左边界与右边界的配合：lower_bound 定头、upper_bound 退一格定尾、缺席时返回 [-1,-1]。
- **35. Search Insert Position（搜索插入位置）**：问 target 不在时应插入哪个下标。这题就是 `lower_bound` 原题——"第一个 ≥target 的位置"与"插入位置"是同一件事。

照 notebook 原文提醒：以上条目用于知识映射，本章实现遵循教学契约，刷题时以 LeetCode 官方题面与评测为准。

**题面入口：** [704](https://leetcode.com/problems/binary-search/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。